In [15]:
from datetime import datetime

In [16]:
import requests
import json
import pandas as pd
import psycopg2
import hashlib

csv_file_path = "Dati_sensori_aria_dal_2019.csv" 
df = pd.read_csv(csv_file_path)
df.columns = [col.lower() for col in df.columns] # Convert column names to lowercase

# Convert DataFrame to a list of dictionaries
data1 = df.to_dict(orient="records")

url = "https://www.dati.lombardia.it/resource/ib47-atvt.json"
response = requests.get(url)
data2 = response.json()
# print(data2)
with open("sensors.json", "w") as f:
    json.dump(data2, f, indent=2)

In [ ]:
import psycopg2
mydb = psycopg2.connect(
 host="localhost",
 database="DB_2019",
 user="postgres",
 password=""  # Replace with your password
)
c = mydb.cursor()

In [19]:
c.execute('''CREATE TABLE IF NOT EXISTS air_quality
    (idsensore INTEGER,
    data TIMESTAMP,
    valore FLOAT,
    stato TEXT,
    idoperatore INTEGER)''')

c.execute('''
CREATE TABLE IF NOT EXISTS sensors (
    id_sensore INTEGER,
    nome_tipo_sensore TEXT,
    unita_misura TEXT,
    id_stazione INTEGER,
    nome_stazione TEXT,
    quota INTEGER,
    provincia TEXT,
    comune TEXT,
    storico TEXT,
    data_start TIMESTAMP,
    utm_nord INTEGER,
    utm_est INTEGER,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    location GEOMETRY(POINT, 4326)
)
''')

In [ ]:
# for record in data1:
#     c.execute("""
#         INSERT INTO air_quality (idsensore, data, valore, stato, idoperatore)
#         VALUES (%s, %s, %s, %s, %s)
#     """, (
#         record.get("idsensore"),
#         record.get("data"),
#         record.get("valore"),
#         record.get("stato"),
#         record.get("idoperatore")
#     ))

batch_size = len(data1) // 20

for i in range(20):
    start = i * batch_size
    end = (i + 1) * batch_size if i < 19 else len(data1)  # Last batch may be smaller

    batch = data1[start:end]

    values = [
        (
            record.get("idsensore"),
            datetime.strptime(record.get("data"), "%d/%m/%Y %H:%M:%S"),
            record.get("valore"),
            record.get("stato"),
            record.get("idoperatore")
        )
        for record in batch
    ]

    c.executemany("""
        INSERT INTO air_quality (idsensore, data, valore, stato, idoperatore)
        VALUES (%s, %s, %s, %s, %s)
    """, values)

    mydb.commit()
    print(f"Batch {i + 1}/20 is uploaded: {len(batch)} records")

c.execute("DELETE FROM air_quality WHERE valore = -9999")

for record in data2:
    c.execute("""
        INSERT INTO sensors (
            id_sensore,
            nome_tipo_sensore,
            unita_misura,
            id_stazione,
            nome_stazione,
            quota,
            provincia,
            comune,
            storico,
            data_start,
            utm_nord,
            utm_est,
            latitude,
            longitude,
            location
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
                ST_SetSRID(ST_MakePoint(%s, %s), 4326))
    """, (
        record.get("idsensore"),
        record.get("nometiposensore"),
        record.get("unitamisura"),
        record.get("idstazione"),
        record.get("nomestazione"),
        record.get("quota"),
        record.get("provincia"),
        record.get("comune"),
        record.get("storico"),
        record.get("datastart"),
        record.get("utm_nord"),
        record.get("utm_est"),
        record.get("lat"),
        record.get("lng"),
        record["location"]["coordinates"][0], # Longitude
        record["location"]["coordinates"][1]  # Latitude
    ))

Batch 1/20 is uploaded: 145007 records
Batch 2/20 is uploaded: 145007 records
Batch 3/20 is uploaded: 145007 records
Batch 4/20 is uploaded: 145007 records
Batch 5/20 is uploaded: 145007 records
Batch 6/20 is uploaded: 145007 records
Batch 7/20 is uploaded: 145007 records
Batch 8/20 is uploaded: 145007 records
Batch 9/20 is uploaded: 145007 records
Batch 10/20 is uploaded: 145007 records
Batch 11/20 is uploaded: 145007 records
Batch 12/20 is uploaded: 145007 records
Batch 13/20 is uploaded: 145007 records
Batch 14/20 is uploaded: 145007 records
Batch 15/20 is uploaded: 145007 records
Batch 16/20 is uploaded: 145007 records
Batch 17/20 is uploaded: 145007 records
Batch 18/20 is uploaded: 145007 records
Batch 19/20 is uploaded: 145007 records
Batch 20/20 is uploaded: 145014 records


In [21]:
c.execute("""
    ALTER TABLE sensors
    ADD CONSTRAINT sensors_pkey PRIMARY KEY (id_sensore);
""")
c.execute("""
    ALTER TABLE air_quality
    ADD CONSTRAINT foreignkey_idsensore
    FOREIGN KEY (idsensore)
    REFERENCES sensors(id_sensore)
""")

In [22]:
c.execute('''CREATE TABLE average_per_day_each_sensor AS
SELECT
  idsensore,
  DATE(data) AS day,
  AVG(valore) AS average_value
FROM
  air_quality
WHERE
  valore != -9999
GROUP BY
  idsensore, DATE(data)''')


c.execute('''
ALTER TABLE average_per_day_each_sensor
ADD CONSTRAINT fk_idsensore
FOREIGN KEY (idsensore)
REFERENCES sensors(id_sensore);
''')

c.execute('''
CREATE TABLE avg_per_day_by_each_comune_and_pollutant AS
SELECT
  s.nome_tipo_sensore,
  s.provincia,
  s.comune,
  v.day,
  AVG(v.average_value) AS avg_value
FROM
  average_per_day_each_sensor v
JOIN
  sensors s ON v.idsensore = s.id_sensore
GROUP BY
  s.nome_tipo_sensore,
  s.provincia,
  s.comune,
  v.day;
''')

In [23]:
c.execute('''
CREATE TABLE IF NOT EXISTS users (
    id SERIAL PRIMARY KEY,
    username VARCHAR(50) UNIQUE NOT NULL,
    password_hash TEXT NOT NULL
);
''')


username = 'admin'
password = 'admin'

# Hash the password using SHA-256
password_hash = hashlib.sha256(password.encode()).hexdigest()


c.execute('''
INSERT INTO users (username, password_hash)
VALUES (%s, %s)
ON CONFLICT (username) DO NOTHING;
''', (username, password_hash))

In [24]:
mydb.commit()
c.close()
mydb.close()